In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#**LDA METHOD GENSIM**

**ATAS INI - CONNECT TO GDRIVE**

**READ DATA**

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/dataTA/Data_TA/dataset_TA.csv'
df_test = pd.read_csv(path)
df_test.head()


Cek english (kalo mau run ini, run preprocess dulu)

In [ ]:
from nltk.corpus import wordnet

def english_ratio(text):
    words = text.lower().split()
    if len(words) == 0:
        return 0
    english_words = sum(1 for w in words if wordnet.synsets(w))
    return english_words / len(words)

ratios = df_test['full_text'].apply(english_ratio)

print("Rata-rata rasio Inggris:", ratios.mean())

In [ ]:
from nltk.corpus import wordnet

def english_ratio(text):
    words = text.lower().split()
    if len(words) == 0:
        return 0
    english_words = sum(1 for w in words if wordnet.synsets(w))
    return english_words / len(words)

# Hitung rasio
df_test['english_ratio'] = df_test['full_text'].apply(english_ratio)

# Tentukan threshold (misal 0.3)
threshold = 0.3

# Filter tweet dengan rasio Inggris >= threshold
english_tweets = df_test[df_test['english_ratio'] >= threshold]

# Tampilkan indeks dan kolom full_text saja
english_tweets[['full_text']]

In [ ]:
print("Jumlah tweet dengan dominasi Inggris:", len(english_tweets))
print("Index tweet tersebut:")
print(english_tweets.index.tolist())

**Data preprocessing**

In [ ]:
# Ambil kolom teks saja
text = df_test['full_text'].astype(str)
print("Jumlah tweet:", len(text))
text.head()

In [ ]:
# Cek panjang karakter
text_length = df_test['full_text'].str.len()
print(text_length.describe())

In [ ]:
# Drop missing
df_test = df_test.dropna(subset=['full_text'])

# Ambil kolom teks
text = df_test['full_text'].astype(str)

print("Jumlah tweet:", len(text))
print("Contoh data:")
print(text.head())

**2️⃣ Preprocessing (Lowercase, Cleaning, Tokenizing, Stopwords, Lemmatization & Stemming)**

In [ ]:
!pip install Sastrawi

**pakai wordnet** (bukan ini yg di run)

-➡️ Mengubah kata ke lemma (bentuk dasar) berdasarkan kamus bahasa Inggris.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

stop_words = set(stopwords.words('indonesian'))
lemmatizer = WordNetLemmatizer()
stemmer = StemmerFactory().create_stemmer()



**Tidak pakai wordnet Untuk english** coba yang ini dulu - ini yang di run

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Download resource yang diperlukan
nltk.download('punkt')
nltk.download('stopwords')

# Stopword Bahasa Indonesia (dasar)
stop_words = set(stopwords.words('indonesian'))

# Stemmer Bahasa Indonesia
stemmer = StemmerFactory().create_stemmer()

**🧩 STEP 1 — Lowercasing**

In [ ]:
# Tambahkan kolom baru langsung di df_test
df_test['lower'] = df_test['full_text'].astype(str).str.lower()

for i in range(5):
    print(f"\n--- Tweet {i+1} ---")
    print("Original :", df_test['full_text'].iloc[i])
    print("Lower    :", df_test['lower'].iloc[i])

df_test.head(15)


In [ ]:
# Ambil kolom lower saja
text1 = df_test['lower'].astype(str)
print("Jumlah tweet:", len(text1))
text1.head()


**🧹 STEP 2 — Cleaning (hapus URL, mention, hashtag, simbol, angka)**

🔹 Tujuan: hapus noise (link, mention, angka, emoji, simbol).
🔹 Setelah ini teks harus lebih “bersih”.

**INI TIDAK DI RUN**

In [ ]:
#versi 1 menghilangkan angka

import re
import pandas as pd

# fungsi cleaning
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)   # hapus URL
    text = re.sub(r"@\w+", "", text)                      # hapus mention
    text = re.sub(r"#\w+", "", text)                      # hapus hashtag
    text = re.sub(r"[^a-zA-Z\s]", " ", text)              # hapus simbol, angka
    text = re.sub(r"\s+", " ", text).strip()              # hapus spasi berlebih
    return text

# terapkan fungsi cleaning
text2 = text1.apply(clean_text)

# gabungkan hasil ke dalam DataFrame untuk perbandingan
compare_df = pd.DataFrame({
    'Before (Lowercase)': text1,
    'After (Cleaned)': text2
})

# tampilkan 5 data pertama dalam tabel
print("🧹 Hasil Cleaning (Before → After):\n")
display(compare_df.head(10))

# tampilkan juga versi print untuk melihat langsung
for i in range(10):
    print(f"\n--- Tweet {i+1} ---")
    print("Before:", text1.iloc[i])
    print("After :", text2.iloc[i])


**ini yang di RUN**

In [ ]:
#tidak menghilangkan angka semantik (5k 10k 21k 42k)

import re
import pandas as pd

def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)   # hapus URL
    text = re.sub(r"@\w+", "", text)                      # hapus mention
    text = re.sub(r"#", "", text)                         # hapus simbol # saja

    # Pertahankan pola seperti 5k, 10k, 21k, 42k
    text = re.sub(r"\b(\d+)(?=[a-zA-Z])", r"\1", text)   # pertahankan angka jika diikuti huruf

    # Hapus simbol selain huruf, angka, dan spasi
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Hapus angka yang berdiri sendiri (misal: 2024, 123)
    text = re.sub(r"\b\d+\b", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# Terapkan cleaning
text2 = text1.apply(clean_text)

# gabungkan hasil ke dalam DataFrame untuk perbandingan
compare_df = pd.DataFrame({
    'Before (Lowercase)': text1,
    'After (Cleaned)': text2
})

# tampilkan 5 data pertama dalam tabel
print("🧹 Hasil Cleaning (Before → After):\n")
display(compare_df.head(10))

# tampilkan juga versi print untuk melihat langsung
for i in range(10):
    print(f"\n--- Tweet {i+1} ---")
    print("Before:", text1.iloc[i])
    print("After :", text2.iloc[i])

**🧩 STEP 3 — Tokenizing**

*   Tujuan: ubah string jadi list kata.
*    Misal "lari pagi seru banget" → ['lari','pagi','seru','banget'].



In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
 # fungsi tokenizing
def tokenize_text(text):
    return nltk.word_tokenize(text)

# terapkan fungsi
text3 = text2.apply(tokenize_text)

# gabungkan hasilnya ke DataFrame buat perbandingan
token_df = pd.DataFrame({
    'After (Cleaned)': text2,
    'After (Tokenized)': text3
})

# tampilkan hasilnya
print("🔤 Hasil Tokenizing (Cleaned → Tokenized):\n")
display(token_df.head(5))

# tampilkan contoh print
for i in range(5):
    print(f"\n--- Tweet {i+1} ---")
    print("Cleaned  :", text2.iloc[i])
    print("Tokenized:", text3.iloc[i])


In [ ]:
# Mengukur panjang dokumen setelah tokenisasi.

print("Rata-rata jumlah token per dokumen:")
print(text3.apply(len).mean())

print("Min token:", text3.apply(len).min())
print("Max token:", text3.apply(len).max())

In [ ]:
# cek dokumen kosong karena min token : 0

empty_docs = text3[text3.apply(len) == 0]

print("Jumlah dokumen kosong:", len(empty_docs))

In [ ]:
empty_index = empty_docs.index

df_test.loc[empty_index, ['lower']].head(5)

In [ ]:
#drop dokumen kosong (empty)

# Simpan index valid (token > 0)
valid_index = text3[text3.apply(len) > 0].index

# Filter semua dataframe
df_test = df_test.loc[valid_index]
text3 = text3.loc[valid_index]

print("Jumlah data setelah drop kosong:", len(text3))

In [ ]:
# Mengukur panjang dokumen setelah tokenisasi.

print("Rata-rata jumlah token per dokumen:")
print(text3.apply(len).mean())

print("Min token:", text3.apply(len).min())
print("Max token:", text3.apply(len).max())

In [ ]:
short_docs = text3[text3.apply(len) < 3]
print("Jumlah dokumen < 3 token:", len(short_docs))

In [ ]:
# Drop final token dibawah 3

valid_index = text3[text3.apply(len) >= 3].index

df_test = df_test.loc[valid_index]
text3 = text3.loc[valid_index]

print("Jumlah data setelah filter <3 token:", len(text3))

**Masukkan penjelasan ini ke skripsi?**

Dokumen dengan jumlah token kurang dari 3 dihapus dari dataset karena dianggap tidak memiliki konteks yang cukup untuk membentuk distribusi topik yang stabil dalam model LDA. LDA merupakan model probabilistik yang mengasumsikan setiap dokumen terdiri atas campuran beberapa topik. Dokumen dengan satu atau dua token cenderung menghasilkan distribusi topik yang tidak stabil serta meningkatkan sparsity dalam korpus. Oleh karena itu, penyaringan ini dilakukan untuk meningkatkan kualitas dan kestabilan model.

**🧯 STEP 4 — Stopword Removal**


*   🔹 Tujuan: buang kata umum yang tidak penting seperti “dan”, “di”, “yang”.



In [ ]:
# ambil stopwords bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# fungsi untuk hapus stopwords
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

# terapkan ke hasil tokenizing
text4 = text3.apply(remove_stopwords)

# buat tabel perbandingan
stopword_df = pd.DataFrame({
    'After (Tokenized)': text3,
    'After (Stopword Removed)': text4
})

# tampilkan tabel hasil
print("🧹 Hasil Stopword Removal (Tokenized → Stopword Removed):\n")
display(stopword_df.head(50))

# tampilkan contoh detail
for i in range(5):
    print(f"\n--- Tweet {i+1} ---")
    print("Tokenized        :", text3.iloc[i])
    print("Stopword Removed :", text4.iloc[i])


In [ ]:
print("Rata-rata token setelah stopword:", text4.apply(len).mean())
print("Min token:", text4.apply(len).min())
print("Max token:", text4.apply(len).max())

In [ ]:
short_docs = text4[text4.apply(len) < 3]
print("Jumlah dokumen < 3 token setelah stopword:", len(short_docs))

In [ ]:
# Drop final token dibawah 3

valid_index = text4[text4.apply(len) >= 3].index

df_test = df_test.loc[valid_index]
text4 = text4.loc[valid_index]

print("Jumlah data final setelah stopword filtering:", len(text4))

In [ ]:
# ambil stopwords bawaan
stop_words = set(stopwords.words('indonesian'))

# tambahkan stopwords Inggris
stop_words.update(stopwords.words('english'))

# tambahkan versi informal
custom_stopwords = [
    'yg', 'aja', 'jg', 'bgt', 'banget',
    'ga', 'gak', 'enggak', 'kalo', 'kalau', 'klo',
    'nya', 'sih', 'dong', 'ko', 'loh',
    'ya', 'lho', 'lah', 'buat', 'biar', 'ok', 'oke',
    'udah', 'sudah', 'kayak', 'kayaknya', 'nih', 'tuh', 'deh',
    'gt', 'utk', 'ni', 'nak','amp','dlm','m','a','tp','lo',
    'gue','gua','gw','de','n','b','e','eu','que','r','na','cr',
    'ak','l','p','g','u','w','q','em',
    'kek','krn','jd','dah','da','tu','lg','dr','trs','sm','kl',
    'dll','si','eh','la','te',
    'wkwk','wkwkwk','wkwkw',
    'com','link','co', 'deu', 'meu', 'agora', 'uma', 'jol','pai',
    'je','nao','quero','lovato','faker','loiro','aitor','larry','fiquei',
    'tent','udh','lu','skrg','yuk','yaa','mah','dur','lingorm','adadikompas',
    'trus','el','amo','gara','iya','vc','kak','ngga','h','nge','es'

]

# gabungkan stopwords
stop_words.update(custom_stopwords)

# re-run stopword removal
text4 = text3.apply(lambda tokens: [w for w in tokens if w not in stop_words])

# cek ulang top 10 kata
from collections import Counter
after_custom = [w for tokens in text4 for w in tokens]
print("🔹 Top 10 kata sesudah custom stopword:")
print(Counter(after_custom).most_common(1000))


In [ ]:
print("Rata-rata panjang token sebelum stopword:", text3.apply(len).mean())
print("Rata-rata panjang token sesudah stopword:", text4.apply(len).mean())


In [ ]:
from collections import Counter

# Gabungkan semua token jadi satu list
before = [w for tokens in text3 for w in tokens]
after = [w for tokens in text4 for w in tokens]

print("🔹 Top 10 kata sebelum stopword:")
print(Counter(before).most_common(50))

print("\n🔹 Top 10 kata sesudah stopword:")
print(Counter(after).most_common(50))


In [ ]:
from collections import Counter

print("🔹 50 kata paling jarang sebelum stopword:")
print(Counter(before).most_common()[-5000:])

print("\n🔹 50 kata paling jarang sesudah stopword:")
print(Counter(after).most_common()[-5000:])

In [ ]:
#cek dan hitung total vocabulary size

# Flatten semua token jadi satu list
all_tokens = [word for tokens in text4 for word in tokens]

# Hitung unique vocabulary
vocab_before_stemming = set(all_tokens)

print("Jumlah total token:", len(all_tokens))
print("Jumlah kata unik (vocabulary size):", len(vocab_before_stemming))

In [ ]:
#ini yang lama

#cek dan hitung total vocabulary size

# Flatten semua token jadi satu list
all_tokens = [word for tokens in text4 for word in tokens]

# Hitung unique vocabulary
vocab_before_stemming = set(all_tokens)

print("Jumlah total token:", len(all_tokens))
print("Jumlah kata unik (vocabulary size):", len(vocab_before_stemming))

In [ ]:
from collections import Counter

token_counts = Counter(all_tokens)

print("Top 20 kata:")
print(token_counts.most_common(1000))

In [ ]:
from collections import Counter

token_counts = Counter(all_tokens)

print("Top 20 kata paling sedikit:")
print(token_counts.most_common()[-20:])

**🌿 STEP 5 — Stemming (Sastrawi)**



*   🔹 Tujuan: ubah kata ke bentuk dasar (contoh: berlari → lari, bagusan → bagus).
*   🔹 Ini langkah terakhir sebelum ke vectorization (TF-IDF / LDA).
   List item



**ini stemming menggunankan wordnet (bahasa inggris)**

In [ ]:
# inisialisasi alat
lemmatizer = WordNetLemmatizer()
stemmer = StemmerFactory().create_stemmer()

# Kamus custom untuk memperbaiki hasil yang tidak sesuai
custom_stem_dict = {
    "perasaan": "perasaan",
    "berlari": "lari",
    "pelari": "lari",
    "bermain": "main",
    "berbagi": "berbagi",
    "olahraganya": "olahraga",
    "larii": "lari",
    "bbrp" : "beberapa",
    "plg" : "pulang",
    "impian": "mimpi",
    "bermimpi" : "mimpi",
    "semangatin" : "semangat",
    "kuatin" : "kuat",
    "lagiii" : "lagi",
    "lair" : "lari",
    "km" : "kilometer"

}

# Tambahan kata baru (bisa ditambah kapan pun)
custom_stem_dict.update({
    "kesalahan": "salah",
    "perjuangan": "juang",
    "semangatnya": "semangat"
})


# fungsi untuk gabungkan keduanya
def lemmatize_and_stem(tokens):
    hasil = []
    for w in tokens:
        # lemmatize dulu (kalau ada kata Inggris)
        lemma = lemmatizer.lemmatize(w)
        # lalu stemming (khusus kata Indonesia)
        stem = stemmer.stem(lemma)
        hasil.append(stem)
    return hasil

# terapkan ke hasil stopword removal
text5 = text4.apply(lemmatize_and_stem)

**ini tidak menggunakan wordnet** - fokus ke bahasa indonesia

In [ ]:
# inisialisasi alat
stemmer = StemmerFactory().create_stemmer()

# Kamus custom untuk memperbaiki hasil yang tidak sesuai
custom_stem_dict = {
    "perasaan": "perasaan",
    "berlari": "lari",
    "pelari": "lari",
    "bermain": "main",
    "berbagi": "berbagi",
    "olahraganya": "olahraga",
    "larii": "lari",
    "bbrp" : "beberapa",
    "plg" : "pulang",
    "impian": "mimpi",
    "bermimpi" : "mimpi",
    "semangatin" : "semangat",
    "kuatin" : "kuat",
    "lagiii" : "lagi",
    "lair" : "lari",
    "km" : "kilometer"

}

# Tambahan kata baru (bisa ditambah kapan pun)
custom_stem_dict.update({
    "kesalahan": "salah",
    "perjuangan": "juang",
    "semangatnya": "semangat"
})


# fungsi
def stem_tokens(tokens):
    hasil = []
    for w in tokens:
        # cek custom dict dulu
        if w in custom_stem_dict:
            hasil.append(custom_stem_dict[w])
        else:
            hasil.append(stemmer.stem(w))
    return hasil

# Terapkan
text5 = text4.apply(stem_tokens)

**RUN YANG INI**

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ===============================
# 1️⃣ Inisialisasi Stemmer
# ===============================
stemmer = StemmerFactory().create_stemmer()

# ===============================
# 2️⃣ Custom Stem Dictionary
# ===============================
custom_stem_dict = {
    # existing
    "perasaan": "perasaan",
    "berlari": "lari",
    "pelari": "lari",
    "bermain": "main",
    "berbagi": "berbagi",
    "olahraganya": "olahraga",
    "larii": "lari",
    "lair": "lari",
    "bbrp": "beberapa",
    "plg": "pulang",
    "impian": "mimpi",
    "bermimpi": "mimpi",
    "semangatin": "semangat",
    "kuatin": "kuat",
    "lagiii": "lagi",
    "km": "kilometer",
    "kesalahan": "salah",
    "perjuangan": "juang",
    "semangatnya": "semangat",
    "nilai": "gear",
    "gin": "softflask",
    "mo": "lari",
    "want": "barefoot",
    "something": "topi",
    "bkk": "duathlon",
    "wanita": "gender",
    "snbt": "makanan",
    "suruh": "pola",
    "emg": "karbohidrat",
    "lapang": "lapangan",
    "z": "creatine",
    "gapapa": "air",
    "orang2": "hidrasi",
    "ang": "21k",
    "masyarakat": "pelari",
    "buka": "peserta",
    "puasa": "trail",
    "ad": "promosi",
    "por": "olahraga",
    "en": "event",
    "tka": "komunitas",
    "mas": "pelari",
    "jog": "jogging",
    "v": "viral",
    "ai": "aplikasi",
    "like": "latihan",
    "go": "lari",
    "one": "tempo",
    "show": "progres",
    "look": "evaluasi",
    "see": "analisis",
    "got": "hasil",
    "al": "interval",
    "progamming": "fartlek",
    "c": "pace",
    "also": "easy",
    "jan": "stride",
    "emang": "aktivitas",
    "tau": "manfaat",
    "abis": "selesai",
    "pas": "waktu",
    "coba": "latihan",
    "bener": "sehat"
}

# ===============================
# 3️⃣ Ambil Semua Kata Unik
# ===============================
all_tokens = [w for tokens in text4 for w in tokens]
unique_words = set(all_tokens)

print("Jumlah kata unik sebelum stemming:", len(unique_words))

# ===============================
# 4️⃣ Stem SEKALI SAJA per kata unik
# ===============================
stem_map = {}

for w in unique_words:
    if w in custom_stem_dict:
        stem_map[w] = custom_stem_dict[w]
    else:
        stem_map[w] = stemmer.stem(w)

# ===============================
# 5️⃣ Terapkan ke Dokumen
# ===============================
text5 = text4.apply(lambda tokens: [stem_map[w] for w in tokens])

# ===============================
# 6️⃣ Evaluasi Hasil
# ===============================
all_tokens_after = [w for tokens in text5 for w in tokens]

print("Total token setelah stemming:", len(all_tokens_after))
print("Vocabulary setelah stemming:", len(set(all_tokens_after)))

In [ ]:
# tampilkan hasil dalam bentuk tabel perbandingan
lemm_df = pd.DataFrame({
    'Sebelum (Stopword Removed)': text4,
    'Sesudah (Lemmatized & Stemmed)': text5
})

print("🔤 Hasil Lemmatization & Stemming:\n")
display(lemm_df.head(15))

# contoh tampilan detil
for i in range(15):
    print(f"\n--- Tweet {i+1} ---")
    print("Sebelum :", text4.iloc[i])
    print("Sesudah :", text5.iloc[i])


In [ ]:
# ============================================================
# CELL 1 — Cari tweet contoh yang konsisten
# ============================================================
# Tujuan: Menemukan 1 tweet dari dataset yang representatif
# untuk dijadikan contoh tunggal di seluruh tahap preprocessing

import pandas as pd

# ---------- Langkah 1: Coba cari tweet dari contoh di skripsi ----------
# Frasa kunci dari contoh tokenisasi di Bab IV
search_phrase_1 = "rela lari keliling dunia"          # dari contoh tokenisasi
search_phrase_2 = "hiburan saya longrun tipis tipis"  # dari contoh stemming
search_phrase_3 = "mau lari olahraga mulai gerak"     # dari contoh data (Tabel 4.2)

candidate = None
selected_idx = None
search_log = []

for phrase in [search_phrase_1, search_phrase_2, search_phrase_3]:
    match = df_test[df_test['full_text'].str.lower().str.contains(phrase, na=False)]
    search_log.append(f"'{phrase}': {len(match)} hasil ditemukan")
    if len(match) > 0 and candidate is None:
        candidate = match.iloc[0]
        selected_idx = match.index[0]

print("=== LOG PENCARIAN TWEET ===")
for log in search_log:
    print(" -", log)

# ---------- Langkah 2: Jika tidak ditemukan, pilih tweet representatif ----------
if candidate is None:
    print("\n⚠️  Tweet contoh tidak ditemukan secara exact.")
    print("➡️  Memilih tweet representatif berdasarkan kriteria:")

    # Kriteria: panjang sedang (50-200 char), mengandung kata kunci domain lari
    domain_keywords = ['lari', 'olahraga', 'run', 'event', 'jogging', 'marathon']
    mask_keyword = df_test['full_text'].str.lower().apply(
        lambda x: any(kw in str(x) for kw in domain_keywords)
    )
    mask_length = df_test['full_text'].str.len().between(60, 180)
    mask_indonesia = ~df_test['full_text'].str.lower().str.contains(
        r'\b(the|and|is|are|was|were)\b', regex=True, na=False
    )

    candidates_pool = df_test[mask_keyword & mask_length & mask_indonesia]

    if len(candidates_pool) > 0:
        candidate = candidates_pool.iloc[5]   # skip beberapa untuk hindari tweet aneh
        selected_idx = candidates_pool.index[5]
        print(f"   ✓ Ditemukan {len(candidates_pool)} kandidat, dipilih index: {selected_idx}")
    else:
        candidate = df_test.iloc[10]
        selected_idx = df_test.index[10]
        print(f"   ⚠️  Fallback ke index 10")
else:
    print(f"\n✅ Tweet contoh ditemukan! Index: {selected_idx}")

# ---------- Tampilkan tweet terpilih ----------
TWEET_FULL_TEXT = candidate['full_text']
print("\n" + "="*60)
print("📌 TWEET TERPILIH SEBAGAI CONTOH KONSISTEN:")
print("="*60)
print(f"Index  : {selected_idx}")
print(f"Teks   : {TWEET_FULL_TEXT}")
print("="*60)

In [ ]:
# ============================================================
# CELL — Cari 1 tweet representatif untuk contoh preprocessing
# ============================================================

import pandas as pd

# ---------- STEP 1: Coba cari tweet dari contoh skripsi ----------
search_phrases = [
    "rela lari keliling dunia",
    "hiburan saya longrun tipis tipis",
    "mau lari olahraga mulai gerak"
]

candidate = None
selected_idx = None
search_log = []

for phrase in search_phrases:
    match = df_test[df_test['full_text'].str.lower().str.contains(phrase, na=False)]
    search_log.append(f"'{phrase}': {len(match)} hasil ditemukan")
    if len(match) > 0 and candidate is None:
        candidate = match.iloc[0]
        selected_idx = match.index[0]

print("=== LOG PENCARIAN TWEET ===")
for log in search_log:
    print(" -", log)

# ============================================================
# STEP 2: Jika tidak ditemukan → pilih tweet representatif
# ============================================================

if candidate is None:
    print("\n⚠️ Tweet contoh tidak ditemukan secara exact.")
    print("➡️ Memilih tweet representatif (filtered & relevan)...")

    # ---------- Keyword khusus dunia lari ----------
    running_keywords = [
        'lari', 'running', 'run', 'jogging',
        'pace', 'km', 'marathon', 'long run',
        'easy run', 'tempo run', 'latihan', 'olahraga', 'strength'
    ]

    # ---------- Filter 1: mengandung keyword ----------
    mask_keyword = df_test['full_text'].str.lower().apply(
        lambda x: any(kw in str(x) for kw in running_keywords)
    )

    # ---------- Filter 2: panjang teks (lebih representatif) ----------
    mask_length = df_test['full_text'].str.len().between(100, 280)

    # ---------- Filter 3: hindari bahasa Inggris dominan ----------
    mask_indonesia = ~df_test['full_text'].str.lower().str.contains(
        r'\b(the|and|is|are|was|were)\b', regex=True, na=False
    )

    # ---------- Filter 4: hindari noise ----------
    noise_keywords = ['judol', 'slot', 'promo', 'link', 'diskon']
    mask_noise = ~df_test['full_text'].str.lower().apply(
        lambda x: any(nk in str(x) for nk in noise_keywords)
    )

    # ---------- Filter 5: prioritaskan tweet yang ada angka (km/pace) ----------
    mask_numeric = df_test['full_text'].str.contains(r'\d+', na=False)

    # ---------- Gabungkan semua filter ----------
    candidates_pool = df_test[
        mask_keyword & mask_length & mask_indonesia & mask_noise & mask_numeric
    ].copy()

    print(f"Jumlah kandidat setelah filtering: {len(candidates_pool)}")

    if len(candidates_pool) > 0:
        # ---------- Pilih tweet paling representatif (median length) ----------
        candidates_pool['length'] = candidates_pool['full_text'].str.len()
        median_length = candidates_pool['length'].median()
        candidates_pool['diff'] = (candidates_pool['length'] - median_length).abs()

        selected_row = candidates_pool.sort_values('diff').iloc[0]

        candidate = selected_row
        selected_idx = selected_row.name

        print(f"✓ Dipilih index: {selected_idx}")
    else:
        candidate = df_test.iloc[10]
        selected_idx = df_test.index[10]
        print("⚠️ Fallback ke index 10")

# ============================================================
# STEP 3: Output hasil akhir
# ============================================================

TWEET_FULL_TEXT = candidate['full_text']

print("\n" + "="*60)
print("📌 TWEET TERPILIH UNTUK CONTOH PREPROCESSING")
print("="*60)
print(f"Index  : {selected_idx}")
print(f"Teks   : {TWEET_FULL_TEXT}")
print("="*60)

In [ ]:
# ============================================================
# CELL — Cari 1 tweet representatif untuk contoh preprocessing
# ============================================================

import pandas as pd

# ---------- STEP 1: Coba cari tweet dari contoh skripsi ----------
search_phrases = [
    "rela lari keliling dunia",
    "hiburan saya longrun tipis tipis",
    "mau lari olahraga mulai gerak"
]

candidate = None
selected_idx = None
search_log = []

for phrase in search_phrases:
    match = df_test[df_test['full_text'].str.lower().str.contains(phrase, na=False)]
    search_log.append(f"'{phrase}': {len(match)} hasil ditemukan")
    if len(match) > 0 and candidate is None:
        candidate = match.iloc[0]
        selected_idx = match.index[0]

print("=== LOG PENCARIAN TWEET ===")
for log in search_log:
    print(" -", log)

# ============================================================
# STEP 2: Jika tidak ditemukan → pilih tweet representatif
# ============================================================

if candidate is None:
    print("\n⚠️ Tweet contoh tidak ditemukan secara exact.")
    print("➡️ Memilih tweet representatif (filtered & relevan)...")

    # ---------- Keyword khusus dunia lari ----------
    running_keywords = [
        'lari', 'running', 'run', 'jogging',
        'pace', 'km', 'marathon', 'long run',
        'easy run', 'tempo run', 'latihan', 'olahraga', 'strength'
    ]

    # ---------- Filter 1: mengandung keyword ----------
    mask_keyword = df_test['full_text'].str.lower().apply(
        lambda x: any(kw in str(x) for kw in running_keywords)
    )

    # ---------- Filter 2: panjang teks (lebih representatif) ----------
    mask_length = df_test['full_text'].str.len().between(100, 280)

    # ---------- Filter 3: hindari bahasa Inggris dominan ----------
    mask_indonesia = ~df_test['full_text'].str.lower().str.contains(
        r'\b(the|and|is|are|was|were)\b', regex=True, na=False
    )

    # ---------- Filter 4: hindari noise ----------
    noise_keywords = ['judol', 'slot', 'promo', 'link', 'diskon']
    mask_noise = ~df_test['full_text'].str.lower().apply(
        lambda x: any(nk in str(x) for nk in noise_keywords)
    )

    # ---------- Filter 5: prioritaskan tweet yang ada angka (km/pace) ----------
    mask_numeric = df_test['full_text'].str.contains(r'\d+', na=False)

    # ---------- Gabungkan semua filter ----------
    candidates_pool = df_test[
        mask_keyword & mask_length & mask_indonesia & mask_noise & mask_numeric
    ].copy()

    print(f"Jumlah kandidat setelah filtering: {len(candidates_pool)}")

    if len(candidates_pool) > 0:
        # ---------- Pilih tweet paling representatif (median length) ----------
        candidates_pool['length'] = candidates_pool['full_text'].str.len()
        median_length = candidates_pool['length'].median()
        candidates_pool['diff'] = (candidates_pool['length'] - median_length).abs()

        selected_row = candidates_pool.sort_values('diff').iloc[0]

        candidate = selected_row
        selected_idx = selected_row.name

        print(f"✓ Dipilih index: {selected_idx}")
    else:
        candidate = df_test.iloc[10]
        selected_idx = df_test.index[10]
        print("⚠️ Fallback ke index 10")

# ============================================================
# STEP 3: Output hasil akhir
# ============================================================

TWEET_FULL_TEXT = candidate['full_text']

print("\n" + "="*60)
print("📌 TWEET TERPILIH UNTUK CONTOH PREPROCESSING")
print("="*60)
print(f"Index  : {selected_idx}")
print(f"Teks   : {TWEET_FULL_TEXT}")
print("="*60)

In [ ]:
# ============================================================
# CELL — Explore tweet kandidat (random + prioritas panjang)
# ============================================================

import pandas as pd

# ---------- Keyword dunia lari ----------
running_keywords = [
    'lari', 'running', 'run', 'jogging',
    'pace', 'km', 'marathon', 'long run',
    'easy run', 'tempo run', 'latihan', 'olahraga', 'strength'
]

# ---------- Filter 1: mengandung keyword ----------
mask_keyword = df_test['full_text'].str.lower().apply(
    lambda x: any(kw in str(x) for kw in running_keywords)
)

# ---------- Filter 2: panjang teks ----------
mask_length = df_test['full_text'].str.len().between(100, 280)

# ---------- Filter 3: hindari bahasa Inggris dominan ----------
mask_indonesia = ~df_test['full_text'].str.lower().str.contains(
    r'\b(the|and|is|are|was|were)\b', regex=True, na=False
)

# ---------- Filter 4: hindari noise ----------
noise_keywords = ['judol', 'slot', 'promo', 'link', 'diskon']
mask_noise = ~df_test['full_text'].str.lower().apply(
    lambda x: any(nk in str(x) for nk in noise_keywords)
)

# ---------- Filter 5: ada angka (km, pace, dll) ----------
mask_numeric = df_test['full_text'].str.contains(r'\d+', na=False)

# ---------- Gabungkan semua filter ----------
candidates_pool = df_test[
    mask_keyword & mask_length & mask_indonesia & mask_noise & mask_numeric
].copy()

print(f"Total kandidat valid: {len(candidates_pool)}")

# ============================================================
# PRIORITAS: pilih tweet yang lebih panjang & informatif
# ============================================================

candidates_pool['length'] = candidates_pool['full_text'].str.len()

# Ambil top 30% tweet terpanjang
top_percent = 0.3
top_n = int(len(candidates_pool) * top_percent)

candidates_pool = candidates_pool.sort_values(
    by='length', ascending=False
).head(top_n)

print(f"Kandidat setelah prioritas panjang: {len(candidates_pool)}")

# ============================================================
# RANDOM SAMPLING (hasil berbeda tiap run)
# ============================================================

sample_n = 7  # bisa ubah (5–10 ideal)
sampled = candidates_pool.sample(n=min(sample_n, len(candidates_pool)), random_state=None)

print("\n" + "="*70)
print("📌 KANDIDAT TWEET (PILIH SALAH SATU INDEX)")
print("="*70)

for idx, row in sampled.iterrows():
    print(f"\nIndex : {idx}")
    print(f"Panjang: {len(row['full_text'])} karakter")
    print(f"Teks  : {row['full_text']}")
    print("-"*70)

In [ ]:
# ============================================================
# CELL — Explore tweet kandidat (random + prioritas panjang)
# ============================================================

import pandas as pd

# ---------- Keyword dunia lari ----------
running_keywords = [
    'lari', 'running', 'run', 'jogging',
    'pace', 'km', 'marathon', 'long run',
    'easy run', 'tempo run', 'latihan', 'olahraga', 'strength'
]

# ---------- Filter 1: mengandung keyword ----------
mask_keyword = df_test['full_text'].str.lower().apply(
    lambda x: any(kw in str(x) for kw in running_keywords)
)

# ---------- Filter 2: panjang teks ----------
mask_length = df_test['full_text'].str.len().between(100, 280)

# ---------- Filter 3: hindari bahasa Inggris dominan ----------
mask_indonesia = ~df_test['full_text'].str.lower().str.contains(
    r'\b(the|and|is|are|was|were)\b', regex=True, na=False
)

# ---------- Filter 4: hindari noise ----------
noise_keywords = ['judol', 'slot', 'promo', 'link', 'diskon']
mask_noise = ~df_test['full_text'].str.lower().apply(
    lambda x: any(nk in str(x) for nk in noise_keywords)
)

# ---------- Filter 5: ada angka (km, pace, dll) ----------
mask_numeric = df_test['full_text'].str.contains(r'\d+', na=False)

# ---------- Gabungkan semua filter ----------
candidates_pool = df_test[
    mask_keyword & mask_length & mask_indonesia & mask_noise & mask_numeric
].copy()

print(f"Total kandidat valid: {len(candidates_pool)}")

# ============================================================
# PRIORITAS: pilih tweet yang lebih panjang & informatif
# ============================================================

candidates_pool['length'] = candidates_pool['full_text'].str.len()

# Ambil top 30% tweet terpanjang
top_percent = 0.3
top_n = int(len(candidates_pool) * top_percent)

candidates_pool = candidates_pool.sort_values(
    by='length', ascending=False
).head(top_n)

print(f"Kandidat setelah prioritas panjang: {len(candidates_pool)}")

# ============================================================
# RANDOM SAMPLING (hasil berbeda tiap run)
# ============================================================

sample_n = 7  # bisa ubah (5–10 ideal)
sampled = candidates_pool.sample(n=min(sample_n, len(candidates_pool)), random_state=None)

print("\n" + "="*70)
print("📌 KANDIDAT TWEET (PILIH SALAH SATU INDEX)")
print("="*70)

for idx, row in sampled.iterrows():
    print(f"\nIndex : {idx}")
    print(f"Panjang: {len(row['full_text'])} karakter")
    print(f"Teks  : {row['full_text']}")
    print("-"*70)

In [ ]:
selected_idx =2644
TWEET_FULL_TEXT = df_test.loc[selected_idx, 'full_text']

print(TWEET_FULL_TEXT)

In [ ]:
# ============================================================
# CELL 2 — Pipeline preprocessing step-by-step untuk 1 tweet
# ============================================================
# Menampilkan transformasi teks dari asli → setiap tahap preprocessing
# Menggunakan fungsi yang SAMA dengan pipeline utama notebook

import re, nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# --- Pastikan variabel pipeline tersedia ---
# Jika ada error, jalankan ulang cell pipeline utama terlebih dahulu

# =========================================================
# TAHAP 0 — Teks Asli (Original)
# =========================================================
step0_original = TWEET_FULL_TEXT
print("=" * 65)
print("TAHAP 0 │ TEKS ASLI (ORIGINAL FULL TEXT)")
print("=" * 65)
print(step0_original)


In [ ]:
# =========================================================
# TAHAP 1 — Lowercasing
# =========================================================
step1_lower = step0_original.lower()

print("\n" + "=" * 65)
print("TAHAP 1 │ LOWERCASING")
print("=" * 65)
print(step1_lower)

In [ ]:
# =========================================================
# TAHAP 2 — Cleaning (tanpa menghapus angka semantik)
# =========================================================
import re

def clean_text_demo(text):
    # ---------- hapus URL ----------
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # ---------- hapus mention ----------
    text = re.sub(r"@\w+", "", text)

    # ---------- hapus simbol hashtag (# saja, bukan katanya) ----------
    text = re.sub(r"#", "", text)

    # ---------- normalisasi format angka + satuan ----------
    # contoh: 5km -> 5 km
    text = re.sub(r"(\d+)(km|k|m|menit|detik|jam)", r"\1 \2", text, flags=re.IGNORECASE)

    # ---------- pertahankan angka & huruf ----------
    text = re.sub(r"[^a-zA-Z0-9\s:]", " ", text)

    # ---------- pertahankan format pace (misal 6:30) ----------
    # (sudah aman karena ':' dipertahankan)

    # ---------- rapikan spasi ----------
    text = re.sub(r"\s+", " ", text).strip()

    return text


step2_clean = clean_text_demo(step1_lower)

print("\n" + "=" * 65)
print("TAHAP 2 │ CLEANING (angka dipertahankan)")
print("=" * 65)
print(step2_clean)

In [ ]:
# =========================================================
# TAHAP 3 — Tokenisasi
# =========================================================
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

step3_tokens = nltk.word_tokenize(step2_clean)

print("\n" + "=" * 65)
print("TAHAP 3 │ TOKENISASI")
print("=" * 65)
print(step3_tokens)

In [ ]:
# =========================================================
# TAHAP 4 — Penghapusan Stopword
# =========================================================
nltk.download('stopwords', quiet=True)

# Gunakan stopwords YANG SAMA dengan pipeline utama
_stop = set(stopwords.words('indonesian'))
_stop.update(stopwords.words('english'))

custom_stopwords_demo = [
    'yg','aja','jg','bgt','banget','ga','gak','enggak','kalo','kalau','klo',
    'nya','sih','dong','ko','loh','ya','lho','lah','buat','biar','ok','oke',
    'udah','sudah','kayak','kayaknya','nih','tuh','deh','gt','utk','ni','nak',
    'amp','dlm','m','a','tp','lo','gue','gua','gw','de','n','b','e','eu','que',
    'r','na','cr','ak','l','p','g','u','w','q','em','kek','krn','jd','dah','da',
    'tu','lg','dr','trs','sm','kl','dll','si','eh','la','te','wkwk','wkwkwk',
    'wkwkw','com','link','co','deu','meu','agora','uma','jol','pai','je','nao',
    'quero','lovato','faker','loiro','aitor','larry','fiquei','tent'
]
_stop.update(custom_stopwords_demo)

step4_no_stop = [w for w in step3_tokens if w not in _stop]

print("\n" + "=" * 65)
print("TAHAP 4 │ STOPWORD REMOVAL")
print("=" * 65)
print(step4_no_stop)

# Tampilkan kata yang dihapus
removed = [w for w in step3_tokens if w in _stop]
print(f"\n📋 Kata yang dihapus ({len(removed)} kata): {removed}")

In [ ]:
# =========================================================
# TAHAP 5 — Stemming (Sastrawi)
# =========================================================
_stemmer = StemmerFactory().create_stemmer()

# Custom stem dict SAMA dengan pipeline utama
custom_stem_dict_demo = {
    "perasaan":"perasaan", "berlari":"lari", "pelari":"lari",
    "bermain":"main", "berbagi":"berbagi", "olahraganya":"olahraga",
    "larii":"lari", "lair":"lari", "bbrp":"beberapa",
    "semangatin":"semangat", "kuatin":"kuat",
    "berlari":"lari", "kebutuhan":"butuh"
}

step5_stemmed = []
for w in step4_no_stop:
    if w in custom_stem_dict_demo:
        step5_stemmed.append(custom_stem_dict_demo[w])
    else:
        step5_stemmed.append(_stemmer.stem(w))

print("\n" + "=" * 65)
print("TAHAP 5 │ STEMMING (Sastrawi)")
print("=" * 65)
print(step5_stemmed)


In [ ]:
# =========================================================
# RINGKASAN TABEL — Semua tahap dalam satu tampilan
# =========================================================
print("\n" + "=" * 65)
print("📊 RINGKASAN PIPELINE PREPROCESSING (1 TWEET)")
print("=" * 65)

summary = {
    "Tahap"   : ["0 – Teks Asli", "1 – Lowercasing", "2 – Cleaning",
                 "3 – Tokenisasi", "4 – Stopword Removal", "5 – Stemming"],
    "Output"  : [
        step0_original[:80] + "..." if len(step0_original) > 80 else step0_original,
        step1_lower[:80] + "..." if len(step1_lower) > 80 else step1_lower,
        step2_clean[:80] + "..." if len(step2_clean) > 80 else step2_clean,
        str(step3_tokens[:8]) + ("..." if len(step3_tokens) > 8 else ""),
        str(step4_no_stop[:8]) + ("..." if len(step4_no_stop) > 8 else ""),
        str(step5_stemmed[:8]) + ("..." if len(step5_stemmed) > 8 else ""),
    ],
    "Jml Token": ["-", "-", "-",
                  len(step3_tokens), len(step4_no_stop), len(step5_stemmed)]
}

summary_df = pd.DataFrame(summary)
display(summary_df)

print(f"\n✅ Index tweet yang digunakan: {selected_idx}")
print(f"✅ Simpan index ini untuk referensi di Bab IV: selected_idx = {selected_idx}")

tidak perlu

**🧩 Kode Step 5 — Gabungkan Token ke Kalimat**

In [ ]:
# Gabungkan list token jadi satu string
df_test['final_text'] = text5.apply(lambda x: ' '.join(x))

# Tampilkan contoh hasil (5 data pertama)
for i in range(5):
    print(f"\n--- Tweet {i+1} ---")
    print("Sebelum :", text4.iloc[i])
    print("Sesudah :", df_test['final_text'].iloc[i])

# Buat tabel perbandingan hasil
hasil_df = pd.DataFrame({
    'Sebelum': text4.head(10),
    'Sesudah': df_test['final_text'].head(10)
})
hasil_df


**🧩 Step 6 — Persiapan Data untuk LDA**



*   Tujuan:
Mengubah teks hasil preprocessing (df['final_text']) menjadi format yang bisa diproses oleh LDA model dari gensim.



In [ ]:
!pip install gensim

**code lama (jangan di run)**

In [ ]:
from gensim import corpora
from gensim.models import LdaModel

# Tokenize ulang (karena LDA butuh list of tokens, bukan string)
tokenized_texts = [text.split() for text in df_test['final_text']]

# Buat dictionary dan corpus
dictionary = corpora.Dictionary(tokenized_texts)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

print(f"Jumlah dokumen: {len(corpus)}")
print(f"Jumlah kata unik: {len(dictionary)}")

# Cek contoh
print("\nContoh dictionary (kata -> id):")
print(list(dictionary.items())[:10])

print("\nContoh corpus (BoW dokumen pertama):")
print(corpus[0])


**code terbaru**

In [ ]:
#Running ini dulu

text5 = text5.loc[df_test.index]

print("len df_test:", len(df_test))
print("len text5:", len(text5))

In [ ]:
from gensim import corpora

# ====================================
# 1️⃣ Buat Dictionary dari text5
# ====================================
dictionary = corpora.Dictionary(text5)

print("Jumlah dokumen:", len(text5))
print("Jumlah kata unik sebelum filtering:", len(dictionary))

# ====================================
# 2️⃣ Filter Extremes (REKOMENDASI)
# ====================================
dictionary.filter_extremes(
    no_below=5,     # kata minimal muncul di 5 dokumen
    no_above=0.5    # hapus kata muncul di >50% dokumen
)

print("Jumlah kata unik setelah filtering:", len(dictionary))

# ====================================
# 3️⃣ Buat Corpus (Bag-of-Words)
# ====================================
corpus = [dictionary.doc2bow(text) for text in text5]

print("\nContoh dictionary (kata -> id):")
print(list(dictionary.items())[:10])

print("\nContoh corpus (BoW dokumen pertama):")
print(corpus[0])

**jangan run ini**

In [ ]:
#perbaikan rebuild ulang
from gensim import corpora

dictionary = corpora.Dictionary(text5)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in text5]

print("len corpus:", len(corpus))

**🚀 Step 7: Latih Model LDA dan Lihat Topik**

mencoba memahami lagi terkait dengan penerapan parameternya bagaimana, terus kemudian pahami rumus lda nya sendiri bagaimana - **TO DO**

In [ ]:
import gensim
print(gensim.__version__)

**code lama** jangan di run

In [ ]:
from gensim.models import LdaModel

# Jumlah topik bisa disesuaikan (misal 3 untuk testing)
num_topics = 10

# Training model LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=20,          # jumlah iterasi per dokumen
    alpha='auto',
    per_word_topics=True
)

# Tampilkan hasil topik
for idx, topic in lda_model.print_topics(-1):
    print(f"\n🔹 Topik {idx+1}:")
    print(topic)


**Code baru** (cuma test aja karena k topiknya blm di eksplor)

In [ ]:
from gensim.models import LdaModel

num_topics = 10

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=20,
    iterations=200,
    alpha='auto',
    eta='auto'
)

# Tampilkan topik
for idx, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"\n🔹 Topik {idx+1}:")
    print(topic)

interpretasi : kenapa iterasinya 200 dan sebagai macamnya

**Eksplorasi K topik** - model ini yang di RUN

In [ ]:
from gensim.models import LdaModel, CoherenceModel
import matplotlib.pyplot as plt

# ===============================
# 1️⃣ Fungsi eksplorasi K
# ===============================
def compute_coherence_values(dictionary, corpus, texts, start=5, limit=21, step=1):
    coherence_values = []
    model_list = []
    k_values = list(range(start, limit, step))

    for k in k_values:
        print(f"Training LDA untuk K={k}...")

        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=42,
            passes=20,
            iterations=200,
            alpha='auto',
            eta='auto'
        )

        model_list.append(model)

        coherencemodel = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )

        coherence = coherencemodel.get_coherence()
        coherence_values.append(coherence)

        print(f"K={k}, Coherence={coherence:.4f}\n")

    return model_list, coherence_values, k_values

**Alasan pemilihan K topik = 20**
Alasan Metodologis (yang biasanya dipakai di skripsi NLP)
1. Menghindari jumlah topik yang terlalu kecil

Jika jumlah topik terlalu sedikit, maka beberapa tema yang berbeda dapat tergabung dalam satu topik sehingga interpretasi topik menjadi kurang jelas.

2. Menghindari jumlah topik yang terlalu banyak

Jika jumlah topik terlalu besar, maka model cenderung menghasilkan topik yang sangat spesifik atau bahkan redundan sehingga sulit diinterpretasikan.

3. Rentang eksplorasi yang umum digunakan

Dalam banyak penelitian topic modeling, eksplorasi jumlah topik biasanya dilakukan dalam rentang 5–20 topik karena rentang tersebut cukup untuk menangkap variasi tema dalam dataset berukuran sedang.

4. Pertimbangan ukuran dataset

Dataset penelitianmu berisi sekitar 4.890 tweet setelah proses deduplikasi, sehingga rentang jumlah topik hingga 20 topik masih dianggap proporsional untuk menangkap keragaman topik dalam korpus.


In [ ]:
# ===============================
# 2️⃣ Jalankan eksplorasi
# ===============================
model_list, coherence_values, k_values = compute_coherence_values(
    dictionary=dictionary,
    corpus=corpus,
    texts=text5,
    start=5,
    limit=21,
    step=1
)

**INI TIDAK PERLU**

**Train 3 model cv tertinggi**
karena :

Kenapa Tidak Langsung Pilih K=5?

Karena:

K kecil → topik lebih umum

Bisa terlalu broad

Bisa kehilangan detail sosial penting

Ingat dataset kamu:

4.516 dokumen

Domain cukup variatif (event, sepatu, fomo, lifestyle, dll.)

K=5 mungkin terlalu kasar.

**TIDAK PERLU TRAIN MODEL** untuk k = 9

In [ ]:
for k in [14]:
    print(f"\n=== K={k} ===")
    model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=20,
        iterations=200,
        alpha='auto',
        eta='auto'
    )

    for idx, topic in model.print_topics(num_topics=k, num_words=10):
        print(f"Topik {idx+1}: {topic}")

Meskipun nilai coherence tertinggi diperoleh pada K=5 sebesar 0.4987, model dengan K=11 dipilih karena memberikan pemisahan topik yang lebih jelas dan interpretatif. Model K=11 menunjukkan struktur tematik yang lebih kaya dan minim overlap, sehingga lebih representatif dalam menggambarkan variasi percakapan olahraga lari di media sosial X.

In [ ]:
# ===============================
# 3️⃣ Tampilkan hasil akhir
# ===============================
print("\n=== HASIL AKHIR COHERENCE ===")
for k, score in zip(k_values, coherence_values):
    print(f"K={k}, Coherence={score:.4f}")

In [ ]:
# ===============================
# 4️⃣ Plot Coherence (REVISED FINAL)
# ===============================
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))

# ===============================
# plot garis utama (lebih tebal & jelas)
# ===============================
plt.plot(k_values, coherence_values,
         marker='o',
         markersize=6,
         linewidth=2.5)

# ===============================
# 🔥 ambil nilai optimal (otomatis)
# ===============================
best_idx = coherence_values.index(max(coherence_values))
best_k = k_values[best_idx]
best_score = coherence_values[best_idx]

# ===============================
# 🔴 highlight titik optimal (WAJIB)
# ===============================
plt.scatter(best_k, best_score,
            s=150,            # lebih besar
            color='red',      # warna kontras
            zorder=5)

# anotasi optimal (format baru)
plt.text(best_k, best_score + 0.002,
         f"Optimal K = {best_k} ({best_score:.3f})",
         ha='center',
         fontsize=10,
         fontweight='bold')

# ===============================
# label & title (FULL ENGLISH)
# ===============================
plt.xlabel("Number of Topics (K)", fontsize=11)
plt.ylabel("Coherence Score (C_v)", fontsize=11)
plt.title("Coherence Score for Determining the Optimal Number of Topics", fontsize=12)

# ===============================
# grid lebih halus
# ===============================
plt.grid(True, linestyle="--", alpha=0.3)

# batas sumbu
plt.xlim(min(k_values)-0.5, max(k_values)+0.5)
plt.ylim(min(coherence_values)-0.01, max(coherence_values)+0.01)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# 5️⃣ Ambil model terbaik
# ===============================
best_k = 14
best_model = model_list[k_values.index(best_k)]

print("Final K yang dipilih:", best_k)

Meskipun nilai coherence tertinggi diperoleh pada K=5 sebesar 0.4987, model dengan K=11 dipilih karena memberikan pemisahan topik yang lebih jelas dan interpretatif. Model K=11 menunjukkan struktur tematik yang lebih kaya dan minim overlap, sehingga lebih representatif dalam menggambarkan variasi percakapan olahraga lari di media sosial X.

In [ ]:
for idx, topic in best_model.print_topics(num_topics=14, num_words=10):
    print(f"\n🔹 Topik {idx+1}:")
    print(topic)

In [ ]:
#ini topiknya menampikan sesuai k yang terpilih

topics = best_model.show_topics(num_topics=14, num_words=10, formatted=False)

for idx, topic in topics:
    words = [word for word, prob in topic]
    print(f"\n🔹 Topik {idx+1}:")
    print(", ".join(words))

**Topic Diversity**

In [ ]:
# Ambil top words dari setiap topik
topics_words = []

for idx, topic in topics:
    words = [word for word, prob in topic]
    topics_words.append(words)

# Flatten semua kata
all_words = [word for topic in topics_words for word in topic]

# Hitung
total_words = len(all_words)
unique_words = len(set(all_words))

topic_diversity = unique_words / total_words

# Output
print("=" * 50)
print("TOPIC DIVERSITY (LDA)")
print("=" * 50)
print(f"Total words  : {total_words}")
print(f"Unique words : {unique_words}")
print(f"Diversity    : {topic_diversity:.4f}")

**# 🔍 INTERPRETASI LDA – K = 11**

---

## 🔹 Topik 1

**Kata kunci:**
training, strength, latih, daily, minum, air, like, real, fat, go

🏷 **Interpretasi:**
👉 **Program Latihan Harian & Asupan Dasar**

Tema ini mencerminkan:

* Latihan rutin (training, strength, latih, daily)
* Pola hidup sederhana seperti minum air
* Konteks kebugaran dan pengelolaan tubuh (fat)

Topik ini mengarah pada percakapan seputar **rutinitas latihan dan kebiasaan dasar kebugaran**, bukan spesifik ke event atau brand.

---

## 🔹 Topik 2

**Kata kunci:**
workout, x, orange, takut, women, pace, life, steps, gara, musim

🏷 **Interpretasi:**
👉 **Motivasi, Pace, dan Konteks Personal**

Tema ini agak lebih sosial dan personal karena muncul kata:

* pace
* steps
* life
* women
* takut

Kemungkinan besar ini menggambarkan percakapan tentang:

* Pengalaman berlari
* Perasaan atau motivasi pribadi
* Konteks kondisi tertentu (musim, rasa takut)

Ini bukan topik brand, tapi lebih ke **pengalaman personal dalam aktivitas olahraga**.

---

## 🔹 Topik 3

**Kata kunci:**
asics, novablast, tim, sepatu, nyaman, blue, tipe, rekomendasi, kaki, cocok

🏷 **Interpretasi:**
👉 **Review & Rekomendasi Sepatu Lari (Asics/Novablast)**

Tema sangat jelas:

* Brand spesifik (Asics, Novablast)
* Kenyamanan (nyaman, cocok)
* Diskusi tipe sepatu
* Rekomendasi

Ini topik **diskusi performa dan pengalaman penggunaan sepatu lari**.

---

## 🔹 Topik 4

**Kata kunci:**
program, nike, one, mari, muscle, back, beneran, langkah, brand, coros

🏷 **Interpretasi:**
👉 **Program Latihan & Brand Perlengkapan**

Mengandung kombinasi:

* Program
* Brand (Nike, Coros)
* Muscle
* Langkah

Kemungkinan ini percakapan tentang:

* Program latihan
* Brand wearable atau perlengkapan
* Performa

Berbeda dari Topik 3 karena lebih ke **ekosistem brand & program**, bukan hanya sepatu.

---

## 🔹 Topik 5

**Kata kunci:**
run, event, olahraga, bagus, nilai, minggu, lomba, jam, giat, gelar

🏷 **Interpretasi:**
👉 **Event & Lomba Lari**

Tema sangat jelas:

* Event
* Lomba
* Run
* Gelar
* Minggu

Ini menggambarkan percakapan tentang:

* Event lari
* Penyelenggaraan lomba
* Aktivitas kompetitif

---

## 🔹 Topik 6

**Kata kunci:**
olahraga, jalan, kaki, sehat, rutin, makan, minggu, pas, abis, badan

🏷 **Interpretasi:**
👉 **Gaya Hidup Sehat & Aktivitas Harian**

Berbeda dari Topik 1 karena:

* Lebih umum
* Tidak spesifik ke training
* Lebih ke keseharian (jalan kaki, makan, badan)

Ini tema **olahraga sebagai gaya hidup sehat**, bukan performa.

---

## 🔹 Topik 7

**Kata kunci:**
day, sepatu, tas, support, beli, time, model, rest, pinggang, wts

🏷 **Interpretasi:**
👉 **Aksesori & Aktivitas Jual-Beli Perlengkapan**

Kata penting:

* beli
* wts (want to sell)
* model
* tas
* sepatu

Ini kuat mengarah ke:

* Transaksi perlengkapan
* Marketplace informal
* Diskusi model produk

Berbeda dari Topik 8 karena lebih ke **jual-beli komunitas**.

---

## 🔹 Topik 8

**Kata kunci:**
sepatu, olahraga, harga, shopee, dapat, running, cek, gym, pria, wanita

🏷 **Interpretasi:**
👉 **Marketplace & Harga Sepatu Olahraga**

Sangat jelas komersial:

* harga
* shopee
* pria/wanita
* cek

Ini topik **perdagangan online dan promosi produk olahraga**.

---

## 🔹 Topik 9

**Kata kunci:**
besok, olahraga, body, kali, semangat, jalan, dipake, ajak, speed, lokal

🏷 **Interpretasi:**
👉 **Motivasi Harian & Aktivitas Spontan**

Tema ini menunjukkan:

* Ajakan olahraga
* Semangat
* Aktivitas besok
* Speed

Kemungkinan besar ini percakapan ringan tentang **ajakan dan semangat berolahraga**.

---

## 🔹 Topik 10

**Kata kunci:**
fomo, orang, ikut, org, liat, gapapa, mas, doang, sehat, ken

🏷 **Interpretasi:**
👉 **Fenomena FOMO dalam Olahraga**

Ini sangat sosial:

* fomo
* ikut
* orang
* liat

Menggambarkan:

* Ikut tren
* Pengaruh sosial
* Motivasi eksternal

Ini topik yang kuat secara sosiologis dan menarik untuk analisis.

---

## 🔹 Topik 11

**Kata kunci:**
olahraga, pake, suka, orang, sepatu, main, pagi, enak, gym, bikin

🏷 **Interpretasi:**
👉 **Pengalaman Umum & Preferensi Olahraga**

Tema ini lebih umum dan natural:

* suka
* enak
* pagi
* gym
* sepatu

Ini menggambarkan **preferensi dan pengalaman personal terhadap aktivitas olahraga**.

---

# 🎯 ANALISIS STRUKTUR BESAR

Kalau dikelompokkan secara makro:

1. 🏋️ Latihan & Program → Topik 1, 4
2. 👟 Sepatu & Brand → Topik 3
3. 🛒 Marketplace & Transaksi → Topik 7, 8
4. 🏁 Event & Lomba → Topik 5
5. ❤️ Gaya Hidup & Motivasi → Topik 6, 9, 11
6. 🌍 Fenomena Sosial → Topik 10
7. 👤 Pengalaman Personal → Topik 2

Ini menunjukkan bahwa percakapan lari di X tidak hanya tentang event, tetapi juga:

* Komersialisasi
* Brand engagement
* Gaya hidup
* Tekanan sosial (FOMO)



**Nilai CV tertinggi kedua** (k topik = 18)

In [ ]:
for idx, topic in secondbest_model.print_topics(num_topics=11, num_words=10):
    print(f"\n🔹 Topik {idx+1}:")
    print(topic)

**🎯 TAHAP BERIKUTNYA:
📊 Analisis Distribusi Dokumen per Topik**

Kenapa ini penting?

Karena:

Kita ingin tahu topik mana paling dominan

Kita ingin tahu proporsi percakapan publik

Ini yang akan jadi insight utama penelitian

Coherence hanya memilih model.
Distribusi topik adalah isi penelitian.

**🚀 STEP 1 — Ambil Topik Dominan per Dokumen**

In [ ]:
# Ambil topik dominan setiap dokumen
doc_topics = []

for bow in corpus:
    topics = best_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    doc_topics.append(dominant_topic)

print("Jumlah dokumen:", len(doc_topics))
print("Contoh 10 topik pertama:", doc_topics[:10])

✅ STEP 2 — Masukkan ke DataFrame

In [ ]:
df_test['dominant_topic'] = doc_topics

df_test[['dominant_topic']].head()

✅ STEP 3 — Hitung Distribusi

In [ ]:
topic_distribution = df_test['dominant_topic'].value_counts().sort_index()

print("Distribusi Dokumen per Topik:\n")
print(topic_distribution)

✅ STEP 4 — Hitung Persentase

In [ ]:
topic_percentage = (topic_distribution / len(df_test)) * 100

print("\nPersentase Dokumen per Topik:\n")
print(topic_percentage.round(2))

Hasil analisis distribusi dokumen menunjukkan bahwa topik paling dominan adalah fenomena FOMO dalam olahraga lari dengan persentase 25.84%. Hal ini menunjukkan bahwa partisipasi dalam aktivitas lari tidak hanya dipengaruhi oleh motivasi kesehatan, tetapi juga oleh faktor sosial dan tren. Selain itu, topik event dan lomba lari (18.31%) serta pembelian sepatu melalui marketplace (18.29%) juga memiliki kontribusi signifikan dalam percakapan publik.

**📊 Ringkasan Distribusi Topik (K = 11)**

---





| Topik | Jumlah | Persentase    |
| ----- | ------ | ------------- |
| 0     | 125    | 2.77%         |
| 1     | 104    | 2.30%         |
| 2     | 209    | 4.63%         |
| 3     | 198    | 4.38%         |
| 4     | 405    | 8.97%         |
| 5     | 827    | 18.31%        |
| 6     | 205    | 4.54%         |
| 7     | 263    | 5.82%         |
| 8     | 187    | 4.14%         |
| 9     | 826    | 18.29%        |
| 10    | 1167   | **25.84%** 🥇 |

🎯 Topik Paling Dominan

🥇 Topik 10 → 25.84%
🥈 Topik 5 → 18.31%
🥉 Topik 9 → 18.29%

Tiga topik ini saja sudah mencakup:

62.44% percakapan

Artinya mayoritas diskusi terkonsentrasi pada 3 tema besar.

**🔎 Sekarang Kita Cocokkan Dengan Interpretasi Sebelumnya (K=11)**

Berdasarkan output K=11 tadi:

🔹 Topik 10
fomo, orang, ikut, org

👉 Fenomena FOMO dalam olahraga

🔥 Ini paling dominan (25.84%)
Artinya:
Percakapan olahraga lari di X sangat dipengaruhi faktor sosial dan tren.

🔹 Topik 5
run, event, olahraga, lomba

👉 Event & Lomba Lari

🔹 Topik 9
sepatu, olahraga, harga, shopee

👉 Pembelian Sepatu & Marketplace

🧠 Insight Besar Penelitian Kamu

Percakapan olahraga lari di media sosial X:

1️⃣ Didominasi faktor sosial (FOMO)
2️⃣ Diikuti event dan kompetisi
3️⃣ Diikuti konsumsi perlengkapan

Ini sangat menarik secara sosial-ekonomi.

**Visualisasi Jurnal**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ===============================
# Data
# ===============================
topics = [f"Topic {i}" for i in range(1, 15)]

percentages = [
    3.17, 4.34, 5.49, 4.25, 4.47, 14.50, 1.51,
    28.63, 5.14, 10.58, 2.26, 4.43, 8.39, 2.83
]

# ===============================
# DataFrame
# ===============================
df = pd.DataFrame({
    "Topic": topics,
    "Percentage": percentages
})

# Urutkan
df = df.sort_values(by="Percentage", ascending=True)

# ===============================
# Plot
# ===============================
plt.figure(figsize=(10,6))

bars = plt.barh(
    df["Topic"],
    df["Percentage"],
    color="#2C3E50"
)

# ===============================
# Label nilai (SUDAH ADA %)
# ===============================
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.3,
        bar.get_y() + bar.get_height()/2,
        f"{width:.2f}%",   # ← ini yang diubah
        va='center',
        fontsize=9
    )

# ===============================
# Title & Axis
# ===============================
plt.title("Distribution of Documents Across Topics", fontsize=13)

plt.xlabel("Percentage (%)", fontsize=11)
plt.ylabel("Topic Number", fontsize=11)

# ===============================
# Styling
# ===============================
plt.grid(axis='x', linestyle='--', alpha=0.2)

for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()

# ===============================
# Save
# ===============================
plt.savefig("topic_distribution_lda.png", dpi=300)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Data topik dan persentase
topics = [
"Topik 1 – Pembelian Sepatu Running",
"Topik 2 – Aktivitas Olahraga Kebugaran",
"Topik 3 – Preferensi Penggunaan Sepatu",
"Topik 4 – Brand Sepatu Running ASICS",
"Topik 5 – Harga Sepatu Running",
"Topik 6 – Olahraga Pagi dan Jalan Kaki",
"Topik 7 – Interaksi Sosial Olahraga",
"Topik 8 – Fenomena FOMO dalam Olahraga",
"Topik 9 – Event dan Komunitas Lari",
"Topik 10 – Program dan Tren Olahraga Lari",
"Topik 11 – Kegiatan Harian Olahraga",
"Topik 12 – Strength Training Pelari",
"Topik 13 – Latihan Fisik dan Beban",
"Topik 14 – Event Outdoor Running"
]

percentages = [3.17,4.34,5.49,4.25,4.47,14.50,1.51,28.63,5.14,10.58,2.26,4.43,8.39,2.83]

# Buat dataframe
df = pd.DataFrame({
    "Topik": topics,
    "Persentase": percentages
})

# Urutkan dari terbesar
df = df.sort_values(by="Persentase", ascending=True)

plt.figure(figsize=(12,8))

bars = plt.barh(df["Topik"], df["Persentase"])

plt.xlabel("Persentase Dokumen (%)")
plt.title("Distribusi Topik pada Model LDA")

# Tambahkan label nilai di batang
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.3, bar.get_y() + bar.get_height()/2,
             f"{width:.2f}%", va='center')

# Grid ringan
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()

# Simpan gambar
plt.savefig("distribusi_topik_lda.png", dpi=300)

plt.show()

lanjut ke visualisasi

In [ ]:
# 🔇 Hilangkan warning deprecation agar output bersih
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 📦 Install pyLDAvis (hanya perlu sekali)
!pip install pyLDAvis

# 📚 Import library yang dibutuhkan
import gensim
import pyLDAvis
import pyLDAvis.gensim_models


In [ ]:
# Asumsi: model LDA kamu sudah bernama 'lda_model'
# dan corpus + dictionary sudah ada
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(best_model, corpus, dictionary)
vis

**📌 Menampilkan Contoh Dokumen per Topik** - validasi kualitatif

-ni penting karena:

* Membuktikan topik memang sesuai konteks

* Menghindari interpretasi salah

* Menguatkan Bab IV

* Disukai dosen pembimbing

**🎯 STEP 1 — Tampilkan 5 Contoh Tweet per Topik**  - Kita ambil dari kolom lower (yang sudah dibersihkan struktur kasarnya).

In [ ]:
# Ambil distribusi topik lengkap per dokumen
doc_topic_probs = []

for bow in corpus:
    topic_probs = best_model.get_document_topics(bow)
    doc_topic_probs.append(dict(topic_probs))

# Ubah ke DataFrame
topic_prob_df = pd.DataFrame(doc_topic_probs).fillna(0)

# Gabungkan dengan df_test
df_analysis = df_test.copy().reset_index(drop=True)
df_analysis = pd.concat([df_analysis, topic_prob_df], axis=1)

In [ ]:
for topic_num in range(best_k):
    print("\n==============================")
    print(f"🔹 TOPIK {topic_num}")
    print("==============================")

    top_docs = df_analysis.sort_values(by=topic_num, ascending=False).head(5)

    for i, row in top_docs.iterrows():
        print(f"\nProb: {row[topic_num]:.4f}")
        print(row['lower'])

In [ ]:
# 🔇 Hilangkan warning deprecation agar output bersih
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 📦 Install pyLDAvis (hanya perlu sekali)
!pip install pyLDAvis

# 📚 Import library yang dibutuhkan
import gensim
import pyLDAvis
import pyLDAvis.gensim_models


In [ ]:
# Asumsi: model LDA kamu sudah bernama 'lda_model'
# dan corpus + dictionary sudah ada
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)
vis

**🧮 Step 9 — Mengukur Nilai Koherensi (C_v)**

dibawah ini adalah code baru **baru**

**dibawah ini adalah code lama**

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.ERROR)

from gensim.models import CoherenceModel

coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=tokenized_texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_lda = coherence_model_lda.get_coherence()

print(f"📊 Nilai Coherence (C_v): {coherence_lda:.4f}")


In [ ]:
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=tokenized_texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_lda = coherence_model_lda.get_coherence()

print(f"📊 Nilai Coherence (C_v): {coherence_lda:.4f}")

Nilai Coherence (C_v) = 0.4439 artinya model LDA kamu sudah cukup koheren, tapi masih bisa ditingkatkan dengan tuning jumlah topik dan penyesuaian preprocessing.

📖 Interpretasi Nilai 0.4439:
Nilai C_v	Kualitas Topik	Keterangan
* 0.0 – 0.30	Buruk	Topik acak, belum terstruktur
* 0.30 – 0.50	Cukup Baik ✅	Topik mulai terpisah, tapi masih tumpang tindih
* 0.50 – 0.65	Baik	Struktur topik sudah jelas
* diatas 0.65	Sangat Baik	Topik sangat terdefinisi & konsisten

Jadi saat ini:

💬 Model kamu sudah bisa digunakan untuk analisis awal, tapi belum ideal untuk hasil akhir skripsi — masih bisa di-tuning supaya C_v naik.

**⚙️ Step 10 — Tuning Jumlah Topik (Cari Nilai C_v Terbaik)**

* Sekarang kita akan mencoba berbagai jumlah topik (misalnya 2–10) dan hitung nilai C_v untuk masing-masing.
Tujuannya adalah mencari jumlah topik paling optimal (yang menghasilkan C_v tertinggi).

In [ ]:
from gensim.models import LdaModel, CoherenceModel
import numpy as np

def compute_coherence_values(dictionary, corpus, texts, start, limit, step):
    coherence_values = []
    model_list = []

    for num_topics in range(start, limit, step):
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=42,
            passes=20,
            alpha=0.1,         # 🔹 ubah dari 'auto' ke nilai kecil tetap
            eta=0.1,           # 🔹 tambahkan eta untuk kestabilan distribusi kata
            per_word_topics=False  # 🔹 nonaktifkan untuk dataset kecil
        )

        model_list.append(model)

        coherencemodel = CoherenceModel(
            model=model, texts=texts, dictionary=dictionary, coherence='c_v'
        )
        coherence_values.append(coherencemodel.get_coherence())

    return model_list, coherence_values


# Jalankan ulang dengan versi stabil
start, limit, step = 1, 11, 1

model_list, coherence_values = compute_coherence_values(dictionary, corpus, tokenized_texts, start, limit, step)

# Hasilkan nilai coherence
for m, cv in zip(range(start, limit, step), coherence_values):
    print(f"Jumlah Topik = {m}, Coherence Score = {cv:.4f}")


**VISUALISASI UNTUK TAMBAHAN**

**interpretasi** :

Dari grafik di atas, terlihat bahwa nilai Coherence Score cenderung berfluktuasi pada rentang 0.43–0.54.
Nilai tertinggi diperoleh pada jumlah topik = 1 dengan Coherence Score = 0.5391, diikuti oleh jumlah topik = 9 dengan Coherence Score = 0.5373.
Walaupun model dengan satu topik memberikan koherensi paling tinggi, model tersebut cenderung terlalu umum (kurang mampu menangkap variasi tema).
Oleh karena itu, model dengan 9 topik dipilih karena memberikan keseimbangan antara nilai koherensi yang tinggi dan keragaman topik yang lebih kaya secara semantik.

**Visualisasi WORDCLOUD**

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ambil semua topik dari model LDA
topics = best_model.show_topics(
    num_topics=best_model.num_topics,
    num_words=30,
    formatted=False
)

num_topics = len(topics)

cols = 2
rows = (num_topics + 1) // 2

fig, axes = plt.subplots(rows, cols, figsize=(15, rows*5))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < num_topics:
        topic_words = dict(topics[i][1])

        wc = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap='viridis',
            max_words=30,
            random_state=42
        ).generate_from_frequencies(topic_words)

        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f"Topik {i+1}", fontsize=16)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ambil semua topik dari model LDA
topics = best_model.show_topics(
    num_topics=best_model.num_topics,
    num_words=30,
    formatted=False
)

num_topics = len(topics)

cols = 2
rows = (num_topics + 1) // 2

fig, axes = plt.subplots(rows, cols, figsize=(15, rows*5))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < num_topics:
        topic_words = dict(topics[i][1])

        wc = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap='viridis',
            max_words=30,
            random_state=42
        ).generate_from_frequencies(topic_words)

        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f"Topik {i+1}", fontsize=16)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ambil semua topik dari model LDA
topics = best_model.show_topics(num_topics=best_model.num_topics, formatted=False)

# Gabungkan semua kata dari seluruh topik
all_topic_words = {}

for topic in topics:
    for word, prob in topic[1]:
        all_topic_words[word] = all_topic_words.get(word, 0) + prob

# Buat WordCloud
wc = WordCloud(
    width=1000,
    height=600,
    background_color='white',
    colormap='plasma',
).generate_from_frequencies(all_topic_words)

# Visualisasi
plt.figure(figsize=(12,8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud Seluruh Topik LDA", fontsize=18)
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ambil semua topik dari model LDA (lebih banyak kata)
topics = best_model.show_topics(
    num_topics=best_model.num_topics,
    num_words=30,
    formatted=False
)

# Gabungkan semua kata dari seluruh topik
all_topic_words = {}

for topic in topics:
    for word, prob in topic[1]:
        all_topic_words[word] = all_topic_words.get(word, 0) + prob

# Buat WordCloud
wc = WordCloud(
    width=1000,
    height=600,
    background_color='white',
    colormap='plasma',
    max_words=50,
    random_state=42
).generate_from_frequencies(all_topic_words)

# Visualisasi
plt.figure(figsize=(12,8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud Seluruh Topik LDA", fontsize=18)
plt.show()